# 10 — Quatro contas AWS e suas fronteiras

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 4 contas AWS distintas

**Esta aula não executa em uma conta única.** O fluxo mostrado envolve 4 contas (platform, retail-banking, transaction-banking, lending-wealth), mas o manifesto e o IaC atuais exigem as quatro contas preparadas: plataforma, Retail, Transaction e Lending. Sem elas, reserve a aula para demonstração do instrutor no ambiente completo ou retire-a da seleção de execução.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 10](../docs/diagramas/svg/10_santander_multi_conta.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "10")
print({"lab": "10", "dependencies": list(dependencies), "region": config["region"]})

### Confirmar quatro contas reais

Os perfis do operador ou as roles do runner identificam as contas. IDs repetidos não são aceitos como cenário entre contas.

In [ ]:
from santander_demo.aws_workshop.config import DOMAINS, check_accounts
accounts = check_accounts(config, domains=DOMAINS)
assert len({x["account_id"] for x in accounts.values()}) == 4
print(json.dumps(accounts, indent=2))

### Verificar Gateway, Runtime, Policy e Memory em cada domínio

Cada consulta usa a sessão da conta correspondente e o stack exato. Uma aplicação implantada só na plataforma não satisfaz esta verificação.

In [ ]:
environments = {}
for domain in DOMAINS:
    environments[domain] = AWSWorkshop(CONFIG, domain).preflight()
print(json.dumps(environments, indent=2))

**Mapa:** plataforma é o hub C; Retail é a origem A; Transaction é a fonte B; Lending é o terceiro domínio bancário. IAM/STS controlam administração e observação. Os Gateways validam a pessoa por JWT e aplicam suas próprias políticas.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.